In [0]:
# Imports

from pyspark.sql.functions import col, trim, to_date, expr, when, current_timestamp, lower
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col

In [0]:
# Ler a Bronze V2
df_silver = spark.table("workspace.drs_bronze.covid_19")
print(f"Total registros: {df_silver.count()}")


In [0]:
# Padronizando os nomes das colunas
df_covid_19 = df_silver \
    .withColumnRenamed("data_boletim", "bulletin_data") \
    .withColumnRenamed("bairro", "neighborhood") \
    .withColumnRenamed("confirmados", "confirmed") \
    .withColumnRenamed("ativos", "assets") \
    .withColumnRenamed("obitos", "deaths")
 
print(f"Total registros: {df_covid_19.count()}")


In [0]:
# Convertendo campos string para minúsculo
df_covid_19 = df_covid_19 \
    .withColumn("bulletin_data", lower(col("bulletin_data"))) \
    .withColumn("neighborhood",  lower(col("neighborhood")))
 

In [0]:
# Removendo acentos dos campos string
from pyspark.sql.functions import translate

def remove_accents(df, columns):
    accented     = "áàãâäéèêëíìîïóòõôöúùûüçÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇ"
    unaccented   = "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"
    for c in columns:
        df = df.withColumn(c, translate(col(c), accented, unaccented))
    return df

df_covid_19 = remove_accents(df_covid_19, [
    "neighborhood",
    "bulletin_data"
])

display(df_covid_19)

In [0]:
# Criando o campo source_file (já em minúsculo)
from pyspark.sql.functions import col, trim, to_date, expr, when, current_timestamp, lit

df = df_covid_19.withColumn("source_file", lower(lit("covid_19_caso_full.csv")))

In [0]:
# Separando registros inválidos para quarentena
 
df_invalid = df.filter("""
bulletin_data IS NULL
OR neighborhood IS NULL
OR confirmed IS NULL
OR assets IS NULL
OR deaths IS NULL
""")
print(f"Total registros inválidos: {df_invalid.count()}")

In [0]:
# Salvando quarentena da Silver V2

df_invalid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.drs_silver.covid_19_quarantine")


In [0]:
# criando o campo ingestion_timestamp
from pyspark.sql.functions import current_timestamp
 
df = df.withColumn("ingestion_timestamp", current_timestamp()) \
       .withColumn("city",  lower(lit("padre bernardo"))) \
       .withColumn("state", lower(lit("go")))

In [0]:
# Filtrando valores válidos
df_silver_filtered = (
    df.dropDuplicates()
      .filter(
          (col("bulletin_data").isNotNull()) &
          (col("confirmed").isNotNull()) &
          (col("assets").isNotNull()) &
          (col("deaths").isNotNull())
      )
)
print(f"Total registros válidos: {df_silver_filtered.count()}")


In [0]:
# Criando o campo silver_processed_timestamp
df_silver_filtered = df_silver_filtered \
    .withColumn("silver_processed_timestamp", current_timestamp()) \
    .withColumn("created_at", current_timestamp()) \
    .withColumn("updated_at", current_timestamp())

In [0]:
# Garantindo a deduplicação por chave de negócio
 
window_spec = Window.partitionBy(
    "neighborhood",
    "bulletin_data",
    "confirmed",
    "assets",
    "deaths",
).orderBy(
    col("ingestion_timestamp").desc()
)
 
df_silver_deduplicated = (
    df_silver_filtered
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
# Salvando a tabela Silver em uma staging table
 
df_silver_deduplicated.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.drs_silver.covid_19_staging")

In [0]:
df_silver_deduplicated.printSchema()

In [0]:

# Craindo tabela final se não existir

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.drs_silver.covid_19
AS
SELECT * FROM workspace.drs_silver.covid_19_staging WHERE 1 = 0
""")

In [0]:
# Salvando tabela drs_silver_v2 com MERGE

spark.sql("""
MERGE INTO workspace.drs_silver.covid_19 AS target
USING workspace.drs_silver.covid_19_staging AS source
 
ON target.bulletin_data = source.bulletin_data
AND target.neighborhood = source.neighborhood
 
WHEN MATCHED THEN
UPDATE SET
    target.confirmed                    = source.confirmed,
    target.assets                       = source.assets,
    target.city                         = source.city,
    target.state                        = source.state,
    target.deaths                       = source.deaths,
    target.ingestion_timestamp          = source.ingestion_timestamp,
    target.source_file                  = source.source_file,
    target.silver_processed_timestamp   = source.silver_processed_timestamp,
    target.updated_at                   = current_timestamp()
 
WHEN NOT MATCHED THEN
INSERT (
    bulletin_data,
    neighborhood,
    confirmed,
    assets,
    city,
    state,
    deaths,
    ingestion_timestamp,
    source_file,
    silver_processed_timestamp,
    created_at,
    updated_at
)
VALUES (
    source.bulletin_data,
    source.neighborhood,
    source.confirmed,
    source.assets,
    source.city,
    source.state,
    source.deaths,
    source.ingestion_timestamp,
    source.source_file,
    source.silver_processed_timestamp,
    current_timestamp(),
    current_timestamp()
)
""")


In [0]:
# Validando a tabela drs_silver_votacao_secao_2024_go

spark.sql("""
SELECT
COUNT(*) AS total_rows
FROM workspace.drs_silver.covid_19
--GROUP BY 1 
""").display()

In [0]:
# Verificar duplicidade pela chave do merge

spark.sql("""
SELECT
    neighborhood,
    bulletin_data,
    confirmed,
    assets,
    deaths,
    COUNT(*) AS qtd
FROM workspace.drs_silver.covid_19
GROUP BY 1,2,3,4,5 
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""").display()

In [0]:
# Data Quality checks da Silver V2
dq = spark.sql("""
SELECT
    SUM(CASE WHEN neighborhood IS NULL THEN 1 ELSE 0 END) AS null_neighborhood,
    SUM(CASE WHEN bulletin_data IS NULL THEN 1 ELSE 0 END) AS null_bulletin_data,
    SUM(CASE WHEN confirmed IS NULL THEN 1 ELSE 0 END) AS null_confirmed,
    SUM(CASE WHEN assets IS NULL THEN 1 ELSE 0 END) AS null_assets,
    SUM(CASE WHEN deaths IS NULL THEN 1 ELSE 0 END) AS null_deaths
FROM workspace.drs_silver.covid_19
""")

display(dq)

In [0]:
# validação quarentena
spark.sql("""
SELECT COUNT(*) AS total_invalid
FROM workspace.drs_silver.covid_19_quarantine
""").display()

In [0]:
# Falhar notebook se houver erro crítico de qualidade
dq_row = dq.collect()[0]

if (
    dq_row["null_confirmed"] < 0 or
    dq_row["null_assets"] < 0 or
    dq_row["null_deaths"] < 0
):
    raise Exception("Data Quality check failed in workspace.drs_silver.local_votacao")